# 적용한 내용
[페르소나 사전 생성](#temperature-페르소나-사전-생성) 시 [system_message, temperature] 형태를 value로 가지도록 생성한다.
이후 [챗봇 함수 정의](#챗봇-함수-정의)에서 ask 함수를 정의할 때,
기존 llm 객체에 페르소나 사전에 정의된 temperature를 바인딩한 후 chain을 새롭게 만들어 적용한다.
message와 temperature를 list로 묶은 이유는, json 파일로 저장할 때 호환성을 고려한 결과이다.

# 실행 환경 감지

In [2]:
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


# API 키 로드

In [3]:
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


# 패키지 임포트

In [4]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


# llm, prompt, chain 생성 및 작동 확인

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}")
])

chain = prompt | llm   # LCEL 파이프 연산자 (2교시 3부)

# 동작 확인
result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로."})
print(result.content)

이상치는 데이터 집합에서 다른 값들과 현저하게 차이나는 값으로, 일반적인 패턴이나 경향에서 벗어난 데이터를 의미합니다.


# temperature 페르소나 사전 생성

In [ ]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": ["당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.", 0],
    "마케터": ["당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.", 0.6],
    "개발자": ["당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.", 0.2],
    "기획자": ["당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.", 0.4],
    "디자이너": ["당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.", 0.8],
}

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


# 챗봇 함수 정의

In [7]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 및 페르소나 별 temperature 조회
    system_message, persona_temperature = PERSONAS[persona_name]

    # 2. temperature가 반영된 모델 생성 및 체인 갱신
    chain = prompt | llm.bind(temperature=persona_temperature)

    # 3. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 4. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [8]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 챗봇 실행 함수 정의

In [9]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

# 챗봇 실행

In [ ]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 페르소나(Persona)는 특정 사용자 집단의 특성과 행동을 대표하는 가상의 인물입니다. 주로 마케팅, UX/UI 디자인, 제품 개발 등에서 사용되며, 고객의 요구와 목표를 이해하고, 그에 맞춘 전략을 수립하는 데 도움을 줍니다.

페르소나를 만들기 위해서는 다음과 같은 데이터가 필요합니다:

1. **인구통계학적 정보**: 나이, 성별, 직업, 교육 수준 등.
2. **행동 패턴**: 구매 습관, 온라인 활동, 브랜드 충성도 등.
3. **목표와 동기**: 제품이나 서비스 사용의 이유, 해결하고자 하는 문제 등.

예를 들어, 한 연구에 따르면, 페르소나를 활용한 마케팅 전략은 전환율을 10-20% 향상시킬 수 있습니다. 이는 고객 맞춤형 접근이 효과적임을 보여줍니다.

[분석가] 페르소나(Persona)는 특정 사용자 집단의 특성과 행동을 대표하는 가상의 인물입니다. 주로 마케팅, UX/UI 디자인, 제품 개발 등에서 사용되며, 고객의 요구와 목표를 이해하는 데 도움을 줍니다. 

페르소나를 만들기 위해서는 다음과 같은 데이터가 필요합니다:

1. **인구통계학적 정보**: 나이, 성별, 직업 등.
2. **행동 패턴**: 구매 습관, 온라인 활동 등.
3. **목표와 필요**: 제품이나 서비스에서 원하는 것.
4. **고충**: 해결하고자 하는 문제나 불만.

예를 들어, 30대 중반의 직장인 여성 페르소나는 "시간이 부족한 바쁜 직장인"으로 설정할 수 있으며, 이 페르소나를 통해 제품 개발이나 마케팅 전략을 수립할 수 있습니다. 페르소나를 활용하면 고객 중심의 접근이 가능해져, 성공적인 비즈니스 결과를 도출할 확률이 높아집니다.

[분석가] 페르소나(Persona)는 특정 사용자 집단의 특성과 행동을 대표하는 가상의 인물입니다. 주로 마케팅, UX/UI 디자인, 제품 개발 등에서 사용되며, 고객의 요구와 목표를 이해하고, 그에 맞춘 전략을 수립하

: 